# NB 62 — Spectral analysis: how many subspace directions does a concept need?

The Subspace-Method (CLAFIC) question on our data: per NB 32 pool concept, how
many principal directions of its whitened cloud capture 80 / 90 / 95 % of the
energy (cumulative eigenvalue ratio)? That is the SM-implied "ideal `r`". We
compute it for **both constructions** and **both language sets**, and compare to
the fixed `RANK = 8` used in NB 56–61.

- **pooled** — SVD of the pooled whitened token features `(N, d)`.
- **hierarchical** — SVD of the stacked per-lemma top-1 directions `(d, m)`.

**Correctness note.** The shipped `SubspaceUnembeddingRepresentation._lemma_feature_blocks`
collapses a synset's stacked `(max_lemmas, max_tokens)` token array into a single
block, so the deployed hierarchical build is silently **rank-1** (verified: all
cached `*_hier_*` bases have one non-zero column). Here we split the stacked
array **per lemma row** (dropping all-pad rows) to measure the *intended*
hierarchical spectrum — the same fix that will go into the module in phase 2.

Languages: **multilingual** (`Llama`, `min_lemmas=11`) and **English** (`Default`,
`min_lemmas=4`). GPU builds the whitened features only — no generation.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from frames.nlp import MultiLingualWordNetSynsets, SupportedLanguages
from subspace import SubspaceUnembeddingRepresentation
from subspace.concept import _principal_directions

In [ ]:
MAX_TOKENS = 3
R_LEMMA = 1
RANK_REF = 8
RTOL = 1e-3
THRESHOLDS = [0.80, 0.90, 0.95]
KMAX = 20
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# (label, language set, min_lemmas)
CONFIGS = [
    ("multilingual", SupportedLanguages.Llama, 11),
    ("english", SupportedLanguages.Default, 4),
]

POOL = json.loads(Path("resources/32_concept_pool_bf16.json").read_text())
CONCEPTS = [r["synset"] for r in POOL["pool"]]
print(f"{len(CONCEPTS)} concepts x {len(CONFIGS)} language sets")

In [ ]:
sur = SubspaceUnembeddingRepresentation.from_model_id(
    MODEL_ID, rank=RANK_REF, r_lemma=R_LEMMA,
    language_codes=SupportedLanguages.Llama,
    torch_dtype=torch.bfloat16, device_map="auto",
)
PAD = sur.model.tokenizer.pad_token_id


def cum_energy(svals):
    e = svals.float() ** 2
    return (e.cumsum(0) / e.sum().clamp_min(1e-12)).cpu().numpy()


def L_for(cum, kappa):
    return int((cum < kappa).sum()) + 1


def curve(cum, kmax):
    out = np.ones(kmax)
    out[: len(cum)] = cum[:kmax]
    return out


def lemma_blocks(tok2d):
    """One whitened feature block per lemma row (all-pad rows dropped)."""
    blocks = []
    for row in np.asarray(tok2d):
        feats = sur._features_for_ids(row)  # (n_i, d), pad already dropped
        if feats.shape[0] > 0:
            blocks.append(feats.float())
    return blocks


all_rows, curves = {}, {}
for label, langs, min_lemmas in CONFIGS:
    data = sur.data if langs == SupportedLanguages.Llama else MultiLingualWordNetSynsets(
        language_codes=langs
    )
    synsets_all = data.get_all_synsets(sur.model.tokenizer, min_lemmas, MAX_TOKENS)
    rows, pooled_c, hier_c = [], [], []
    for name in CONCEPTS:
        sdf = synsets_all[synsets_all["synset"] == name]
        if len(sdf) == 0:
            continue  # concept below the lemma floor in this language set
        tok2d = sdf["tokens"].values[0]

        feats = sur._features_for_ids(tok2d.reshape(-1)).float()  # pooled cloud (N, d)
        sp = torch.linalg.svdvals(feats)
        cp = cum_energy(sp)

        blocks = lemma_blocks(tok2d)
        dirs = [_principal_directions(b, R_LEMMA, RTOL) for b in blocks]
        dirs = [d for d in dirs if d.shape[-1] > 0]
        b_all = torch.cat(dirs, dim=-1)  # (d, m)
        sh = torch.linalg.svdvals(b_all)
        ch = cum_energy(sh)

        pooled_c.append(curve(cp, KMAX))
        hier_c.append(curve(ch, KMAX))
        rows.append({
            "concept": name, "n_tokens": int(feats.shape[0]), "n_lemmas": len(dirs),
            "pooled_L80": L_for(cp, .8), "pooled_L90": L_for(cp, .9), "pooled_L95": L_for(cp, .95),
            "pooled_energy@8": float(cp[min(RANK_REF, len(cp)) - 1]),
            "hier_L80": L_for(ch, .8), "hier_L90": L_for(ch, .9), "hier_L95": L_for(ch, .95),
            "hier_energy@8": float(ch[min(RANK_REF, len(ch)) - 1]),
        })
    df = pd.DataFrame(rows)
    df.to_csv(f"resources/62_spectral_per_concept_{label}.csv", index=False)
    all_rows[label] = df
    curves[label] = {"pooled": np.array(pooled_c), "hier": np.array(hier_c)}
    print(f"[{label}] {len(df)} concepts | n_lemmas median {int(df.n_lemmas.median())}")

In [ ]:
def block(df, method):
    return {
        "L80 med": df[f"{method}_L80"].median(),
        "L90 med": df[f"{method}_L90"].median(),
        "L95 med": df[f"{method}_L95"].median(),
        "L90 mean": round(df[f"{method}_L90"].mean(), 2),
        "energy@8 mean": round(df[f"{method}_energy@8"].mean(), 3),
        "share L90<=8": round((df[f"{method}_L90"] <= 8).mean(), 3),
    }


summary = pd.DataFrame({
    f"{label} / {method}": block(all_rows[label], method)
    for label, _, _ in CONFIGS
    for method in ("pooled", "hier")
}).T
summary.to_csv("resources/62_spectral_summary.csv")
print(summary.to_string())

In [ ]:
xs = np.arange(1, KMAX + 1)
styles = {"multilingual": "-", "english": "--"}
colors = {"pooled": "tab:gray", "hier": "tab:blue"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, method in zip(axes, ("pooled", "hier")):
    for label, _, _ in CONFIGS:
        m = curves[label][method].mean(axis=0)
        ax.plot(xs, m, styles[label], marker="o", ms=3, color=colors[method],
                label=f"{label}")
    for k in THRESHOLDS:
        ax.axhline(k, color="red", ls=":", lw=0.6)
    ax.axvline(RANK_REF, color="green", ls="--", lw=1.0, label=f"r={RANK_REF}")
    ax.set_xlabel("# subspace directions")
    ax.set_ylabel("mean cumulative energy ratio")
    ax.set_title(f"{method}: energy vs #directions")
    ax.legend(fontsize=8)
plt.suptitle("Intrinsic dimensionality of pool concepts (EN vs multilingual)")
plt.tight_layout()
plt.savefig("resources/62_spectral_analysis.png", dpi=150)
plt.show()

## Reading it

- `Lₚ(κ)` is the SM-implied ideal rank for capturing κ of a concept's cloud
  energy; compare medians and `energy@8` against the fixed `r=8`.
- **pooled** counts token-cloud axes, **hierarchical** counts lemma-level axes —
  read each against its own build, not against each other.
- English clouds are more homogeneous (fewer, single-language lemmas) so their
  intrinsic dimensionality should be lower than the multilingual clouds.
- Steering caveat (NB 61): scoring uses `‖Bᵀh‖`, so larger `r` inflates the
  projection and fluency cost — prefer the smallest `r` clearing the knee.

*The deployed hierarchical build is rank-1 (see header); this notebook measures
the intended spectrum. The module fix + experiment reruns are phase 2.*